# TF Serving によるモデルのサービング
## 学習目標
1. Keras モデルを SavedModel 形式でエクスポートする方法を学ぶ
2. SavedModel を読み込んで使用する方法を学ぶ
3. TensorFlow を使用してシグネチャをカスタマイズする方法を学ぶ
4. Agent Platform（Vertex AI）にモデルをデプロイする方法を学ぶ
5. デプロイされたモデルをオンライン推論およびバッチ推論で使用する方法を学ぶ

このラボでは、トレーニング後のモデルをサービング（提供）する方法を学びます。<br>

機械学習モデルのサービングにはインフラストラクチャが必要です。Agent Platform は、セットアップとメンテナンスの手間を軽減するオートスケーリングサービスを提供することで、これをシンプルにします。

Agent Platform を使用するために、Keras モデルを SavedModel 形式でエクスポートし、Agent Platform にデプロイする方法を見ていきます。その過程で、シグネチャ、そのカスタマイズ方法、デプロイされたモデルから予測を取得する方法について学びます。


## セットアップ


In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")



In [ ]:
import base64
import json
import shutil
from datetime import datetime

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from google.cloud import aiplatform
from oauth2client.client import GoogleCredentials

## モデルの構築とトレーニング
このラボではモデルのトレーニング自体は主眼ではないため、シンプルな MobileNet ベースのモデルを作成し、転移学習を使用して迅速にトレーニングしましょう。


In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT + "-flowers"
FILE_DIR = f"gs://{BUCKET}/data"

CLASSES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3

BATCH_SIZE = 32

まだ [create_tfrecords_at_scale.ipynb](https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/notebooks/image_models/solutions/create_tfrecords_at_scale.ipynb) ノートブックを実行していない場合は、下のセルのコメントを解除し、`gs://asl-public-data` バケットからデータをコピーしてください。


In [ ]:
# !gcloud storage buckets create gs://{BUCKET}
# !gcloud storage cp gs://asl-public-data/data/flowers/tfrecords/* {FILE_DIR}

このデータセットには、TFRecord にシリアライズされた花画像が含まれています。tf.data API を使用してデータを読み込み、解析します。


In [ ]:
!gcloud storage ls {FILE_DIR}

In [ ]:
TRAIN_PATTERN = FILE_DIR + "/train*"
EVAL_PATTERN = FILE_DIR + "/eval*"


def parse_example(example):
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)
    example["image"] = tf.io.decode_jpeg(example["image"], channels=3)
    example["image"] = tf.image.resize(
        example["image"], [IMG_HEIGHT, IMG_WIDTH]
    )
    return example["image"], example["label"]


train_ds = (
    tf.data.TFRecordDataset(tf.io.gfile.glob(TRAIN_PATTERN))
    .map(parse_example)
    .batch(BATCH_SIZE)
)
eval_ds = (
    tf.data.TFRecordDataset(tf.io.gfile.glob(EVAL_PATTERN))
    .map(parse_example)
    .batch(10)
)

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)

transfer_model = tf.keras.Sequential(
    [
        keras.Input(
            shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), name="mobilenet_input"
        ),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            len(CLASSES),
            activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(0.0001),
        ),
    ]
)

transfer_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
transfer_model.fit(
    train_ds,
    epochs=5,
    validation_data=eval_ds,
)

## SavedModel のエクスポートと読み込み
トレーニング済みモデルが用意できました。このセクションでは、モデルを SavedModel 形式でエクスポートします。

また、追加のサービング関数を使用してモデルを拡張する方法についても見ていきます。追加のサービング関数を使用すると、モデルの予測前後に前処理や後処理のロジックを組み込むことができます。


### SavedModel 形式でのエクスポート
SavedModel には、トレーニングされたパラメータ（つまり tf.Variable）や計算グラフを含む完全なモデルプログラムが含まれます。SavedModel でエクスポートされたモデルを実行するのに元のモデル構築コードは不要であるため、LiteRT、TensorFlow.js、TensorFlow Serving などで共有・デプロイするのに非常に役立ちます。

以下の API を使用して、SavedModel 形式でモデルを保存および読み込むことができます:

- Keras Model API: Keras は `format="tf_saved_model"` オプションを指定した [keras.Model.export](https://keras.io/api/models/model_saving_apis/export/) メソッドによる SavedModel 形式のエクスポートをサポートしています。また、"onnx" 形式もサポートしています。
- Keras ExportArchive: [ExportArchive](https://keras.io/api/models/model_saving_apis/export/#exportarchive-class) モジュールは、異なるサービングエンドポイントやそのシグネチャの設定など、より詳細な制御をサポートします。

まず、Keras API を使用してトレーニング済みモデルを SavedModel にエクスポートしましょう。


In [ ]:
shutil.rmtree("export", ignore_errors=True)
os.mkdir("export")
# 通常のモデルのシリアライズには、.keras 拡張子を指定して .save() を使用できます
transfer_model.save("export/flowers_model.keras")

transfer_model.export(
    "export/saved_model", format="tf_saved_model", verbose=False
)

ディレクトリの内容を確認してみましょう。


In [ ]:
!ls export/saved_model

ディレクトリ内に複数のファイルがあることが確認できます。

- `saved_model.pb`: 実際の TensorFlow プログラム（モデル）と、テンソル入力を受け取ってテンソル出力を生成する関数を識別する名前付きシグネチャのセットを含む SavedModel のメインファイルです。
- `keras_metadata.pb`: tf.keras.Model.save() 関数でのみ作成されるファイルです。Keras モデルに関するメタデータが含まれます。
- `variables` ディレクトリ: モデルのすべての変数が含まれます。
- `assets` ディレクトリ: SavedModel に必要なアセットと呼ばれる任意のファイルが含まれます。例えば、ルックアップテーブルの初期化に使用される語彙ファイルなどです。読み込み時に、アセットとそれらに依存するシリアライズされた関数は、SavedModel ディレクトリ内の正しいファイルパスを参照します。


### `saved_model_cli` コマンドによる SavedModel の調査

ビルド済みの TensorFlow バイナリを通じて TensorFlow をインストールした場合、SavedModel CLI はすでにシステムの `bin/saved_model_cli` にインストールされています。

SavedModel CLI は、SavedModel に対する以下の 2 つのコマンドをサポートしています:

- `show`: SavedModel から利用可能な計算を表示します。
- `run`: SavedModel から計算を実行します。

ここでは、デフォルトのタグ（`serve`）とシグネチャ（`serving_default`）を指定して、`saved_model_cli show` コマンドで SavedModel を調査してみましょう。


**演習**: `saved_model_cli` コマンドを完成させ、`serving_default` シグネチャのモデル情報を取得してください。

**ヒント**: `saved_model_cli show -h` を実行してコマンドのヘルプを確認できます。


In [ ]:
!saved_model_cli show --dir export/saved_model --tag_set serve # TODO: serve タグと serving_default シグネチャを指定してください。

これで具体的な関数の詳細を確認できます。

- 入力形状: 3 チャンネルの 224x224 画像のバッチ（`-1`）、名前は `'mobilenet_input'`。
- 出力形状: 5 つの浮動小数点値（5 種類の花の確率を表す）のバッチ、名前は `'output_*'`。


### 読み込みと予測

SavedModel としてエクスポートすると、プログラム内でモデルを読み込んで使用できます。

Python プログラムの場合、SavedModel の代わりに `keras.models.load_model` を呼び出して `.keras` ファイルを直接読み込むこともできます。

また、予測フェーズ（Web API の場合など）では、トレーニングフェーズで使用した前処理済みの TFRecord データや `(224, 224, 3)` のバッチテンソルをモデルが常に受け取れるとは限りません。<br>
例えば、モデルが画像データへのファイルパスを受け取る場合を考えてみましょう。その場合、モデルを呼び出す前に画像パスを処理する前処理操作を追加する必要があります。


In [ ]:
filenames = [
    "gs://asl-public-data/data/flowers/jpegs/10172567486_2748826a8b.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10386503264_e05387e1f7_m.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10391248763_1d16681106_n.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10712722853_5632165b04.jpg",
    "gs://asl-public-data/data/flowers/jpegs/10778387133_9141024b10.jpg",
    "gs://asl-public-data/data/flowers/jpegs/112334842_3ecf7585dd.jpg",
]


def preprocess(img_bytes):
    img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
    img = keras.ops.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    return img


def read_from_jpegfile(filename):
    img = tf.io.read_file(filename)
    img = preprocess(img)
    return img


serving_model = keras.models.load_model("export/flowers_model.keras")
input_images = [read_from_jpegfile(f) for f in filenames]

f, ax = plt.subplots(1, 6, figsize=(15, 15))
for idx, img in enumerate(input_images):
    ax[idx].imshow(img.numpy())
    batch_image = keras.ops.expand_dims(img, axis=0)
    batch_pred = serving_model.predict(batch_image)
    pred = batch_pred[0]
    pred_label_index = keras.ops.argmax(pred).numpy()
    pred_label = CLASSES[pred_label_index]
    prob = pred[pred_label_index]
    ax[idx].set_title(f"{pred_label} ({prob:.2f})")

### 追加のサービング関数の定義

では、毎回このような定型的な前処理コードを作成する必要があるのでしょうか？<br>
答えは「いいえ」です！カスタムサービングシグネチャを追加することで、これらの追加の前処理や後処理の関数を SavedModel 自体に組み込むことができます。

一度 SavedModel にコンパイルされると、同じグラフをエッジデバイス、C++ コード内、JavaScript プログラム内など、さまざまな環境で実行できます。

このモデルを Web サーバーにデプロイし、Web API 経由で予測を提供すると仮定しましょう。

API クライアントにとって最も使いやすいシグネチャとはどのようなものでしょうか？例えば、画像コンテンツのテンソルを送信してもらう代わりに、GCS 上の JPEG ファイルパスを送信してもらうだけで済むようにできます。<br>
また、5 つの確率のテンソルを返す代わりに、確率から抽出した理解しやすい情報を送り返すことができます。


In [ ]:
def postprocess(pred):
    top_prob = keras.ops.max(pred, axis=[1])
    pred_label_index = keras.ops.argmax(pred, axis=1)
    pred_label = keras.ops.take(
        keras.ops.convert_to_tensor(CLASSES), pred_label_index
    )

    # カスタム出力
    return {
        "probability": top_prob,
        "flower_type_int": pred_label_index,
        "flower_type_str": pred_label,
    }


def predict_from_filename(filenames):

    # カスタム前処理
    input_images = keras.ops.map(read_from_jpegfile, filenames)

    # モデル
    batch_pred = transfer_model(input_images)  # same as model.predict()

    # カスタム後処理
    processed = postprocess(batch_pred)
    return processed

さらに、base64 エンコードされた画像データを受け取って前処理できる別のサービング関数も定義しましょう。<br>
これは、オンライン予測で生の画像データを送信したい場合に役立ちます。


In [ ]:
def predict_from_b64(img_bytes):

    # カスタム前処理
    input_images = keras.ops.map(preprocess, img_bytes)

    # モデル
    batch_pred = transfer_model(input_images)

    # カスタム後処理
    processed = postprocess(batch_pred)
    return processed

ここで、`add_endpoint()` を使用して、これらのグラフを異なるキー名で SavedModel 形式にエクスポートしましょう。
- `serving_default`: SavedModel のデフォルトキー名です。このキーを `predict_from_filename` サービング関数で上書きします。
- 画像のバイト文字列を受け取る `predict_from_b64` を指定して、新しいサービングシグネチャキー `predict_base64` を追加します。


**演習**: 最初の例を参考に、`add_endpoint` メソッドで `fn` と `input_signature` を指定して、バイト文字列を受け取る 2 つ目のエンドポイントを完成させてください。


In [ ]:
export_archive = keras.export.ExportArchive()

export_archive.track(transfer_model)

export_archive.add_endpoint(
    name="serving_default",
    fn=predict_from_filename,
    # この関数は 1 つの文字列値（ファイル名）を受け取ります。
    input_signature=[keras.InputSpec(shape=(None,), dtype="string")],
)

export_archive.add_endpoint(
    name="predict_base64",
    # TODO
)

export_archive.write_out("export/flowers_model_with_signature", verbose=False)

新しい SavedModel のメタデータを確認してみましょう。


**演習**: `saved_model_cli` コマンドを完成させ、`serving_default` シグネチャのモデル情報を取得してください。


In [ ]:
!saved_model_cli show --dir export/flowers_model_with_signature # TODO: serve タグと serving_default シグネチャを指定してください。

これで、新しい `serving_default` がファイルパスを受け取り、3 つのキー（`flower_type_int`、`flower_type_str`、`probability`）を持つ辞書を返すことが確認できます。

この SavedModel を読み込んで使用してみましょう。<br>
Keras では、`TFSMLayer` レイヤーを使用してコンパイル済みの SavedModel を読み込むことができます（TensorFlow では `tf.saved_model.load` を使用して同様の操作が可能です）。

追加の前処理コードや後処理コードを記述する必要がなくなったことに注目してください。


In [ ]:
serving_fn = keras.layers.TFSMLayer(
    "export/flowers_model_with_signature", call_endpoint="serving_default"
)

pred = serving_fn(keras.ops.convert_to_tensor(filenames))

# カスタム出力を表示
for k in pred.keys():
    print(f"{k:15}: {pred[k].numpy()}")

これらの出力は、確率のベクトルよりも API クライアントにとって有用に見えます。


また、`predict_base64` シグネチャも確認してみましょう。


**演習**: `saved_model_cli` コマンドを完成させ、`serve` タグおよび `predict_base64` シグネチャ配下のモデル情報を取得してください。


In [ ]:
!saved_model_cli show --dir export/flowers_model_with_signature # TODO: serve タグと predict_base64 シグネチャを指定してください。

## Agent Platform Prediction（Vertex AI 予測）

これでモデルをデプロイする準備が整いました！

このノートブックでは、スケーラブルな Agent Platform サービスにモデルをデプロイします。
Agent Platform はバッチ推論（Batch Inference）とオンライン予測（Online Prediction）の両方をサポートしています。

まず、SavedModel を Agent Platform にアップロードしましょう。


### Agent Platform Prediction サービスへのモデルのアップロード


In [ ]:
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

REGION = "us-central1"
MODEL_DISPLAYNAME = f"flower_classifier-{TIMESTAMP}"

print(f"MODEL_DISPLAYNAME: {MODEL_DISPLAYNAME}")

# from https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions/pre-built-containers
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai-restricted/prediction/tf_opt-gpu.2-17:latest"
)

まず、SavedModel を GCS バケットにアップロードします。


In [ ]:
!gcloud storage cp -R export/flowers_model_with_signature gs://{BUCKET}/{MODEL_DISPLAYNAME}

Python SDK を使用してモデルをアップロードできます。

ここでは、`display_name`、SavedModel のパスである `artifact_uri`、モデルが動作するコンテナ環境である `serving_container_image_uri`（この場合は事前構築済みコンテナを選択していますが、必要に応じてカスタムコンテナも使用可能）を指定しています。

詳細については、[SDK ドキュメント](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_upload)を参照してください。


In [ ]:
uploaded_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAYNAME,
    artifact_uri=f"gs://{BUCKET}/{MODEL_DISPLAYNAME}",
    serving_container_image_uri=SERVING_CONTAINER_IMAGE_URI,
)

アップロード後、Agent Platform -> Models をクリックして、コンソール UI 上でモデルを確認できます。


### バッチ推論（Batch Inference）

バッチ推論では、大規模なデータセットをモデルに渡して一括で予測を実行できます。<br>

#### 予測ファイルの作成
[バッチ推論](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions)サービスは、JSON Lines、TFRecord、CSV、またはシンプルなテキストファイルリスト形式を受け付けます。

ここでは、[JSON Lines](https://jsonlines.org/) 形式で多数の画像ファイルパスを含むシンプルなデータセットを作成します。


In [ ]:
files = !gcloud storage ls -r gs://asl-public-data/data/flowers/jpegs/*.jpg
print(len(files))

In [ ]:
JSON_FILE = "batch_prediction.jsonl"

with open(JSON_FILE, "w") as f:
    for file in files:
        f.write(json.dumps({"filenames": file}) + "\n")

!head {JSON_FILE}

In [ ]:
!gcloud storage cp {JSON_FILE} {FILE_DIR}

#### バッチ推論ジョブの送信
[`aiplatform.batch_predict()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_batch_predict) 関数を使用してバッチ推論ジョブを呼び出しましょう。

必要に応じてマシンタイプとアクセラレータを指定できる点に注意してください。<br>
これは、限られた時間内に大量のデータを処理したい場合に非常に便利です。


**演習**: [API ドキュメント](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model)を参考に、`batch_predict` 関数を完成させてください。
- `job_display_name`: バッチ推論ジョブの表示名。
- `gcs_source`: データのファイルパス。ここでは JSON Lines ファイルパスを指定します。
- `gcs_destination_prefix`: 結果が保存されるファイルパスのプレフィックス。
- `machine_type`: バッチ推論ジョブを実行する GCE マシンタイプ。`"n1-standard-4"` を指定します。
- `accelerator_type` および `accelerator_count`: 使用するアクセラレータのタイプと数。このジョブには `"NVIDIA_TESLA_T4"` を 1 つ指定します。


In [ ]:
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

JOB_DISPLAY_NAME = "flower_classification_batch"
MACHINE_TYPE = "n1-standard-4"

batch_pred_job = uploaded_model.batch_predict(
    # TODO: パラメータを指定してください。
)

**注: これには約 20 分かかります。完了するまで待つか、オンライン予測のセクションに進んで後で次のセルに戻ってください。ステータスは Agent Platform -> Models -> Batch Inferences ページで確認できます。**


In [ ]:
if batch_pred_job.output_info:
    output_dir = batch_pred_job.output_info.gcs_output_directory
    results = !gcloud storage cat {output_dir}/prediction.results*
    for r in results[:5]:
        r = json.loads(r)
        print(f"filename       : {r['instance']['filenames']}")
        for k in r["prediction"].keys():
            print(f"{k:15}: {r['prediction'][k]}")
        print("*" * 30)
else:
    print(f"This job is still running.")

### オンライン予測（Online Prediction）

オンライン予測オプションでは、モデル専用のエンドポイントを作成し、Web API として利用できます。

[`aiplatform.Model.deploy`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_deploy) 関数を使用してエンドポイントを作成し、モデルをリンクしましょう。ここでもマシンタイプとアクセラレータを指定できます。

**以下のコマンドの実行には約 10 分かかります。**


In [ ]:
endpoint = uploaded_model.deploy(
    machine_type=MACHINE_TYPE,
    accelerator_type="NVIDIA_TESLA_T4",
    accelerator_count=1,
)

デプロイ後、エンドポイントを呼び出して結果を取得できます。<br>
Agent Platform -> Models -> Endpoints にアクセスして、エンドポイントの詳細を確認できます。

ここでは [Python SDK](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_predict) を使用しますが、任意の実装環境からエンドポイントを呼び出すことができる点に注意してください。


In [ ]:
instances = [{"filenames": f} for f in filenames]

In [ ]:
pred = endpoint.predict(instances=instances)

# カスタム出力を表示
for p in pred.predictions:
    for k in p.keys():
        print(f"{k:15}: {p[k]}")
    print("*" * 30)

### 生の画像を使用したオンライン予測

次に、`predict_base64` シグネチャを呼び出して生の画像データを渡してみましょう。<br>
API 経由で生の画像データを送信するには、データを Base64 エンコードし、以下のように `b64` をキーとする JSON オブジェクトにカプセル化する必要があります:

```python
{ "b64": <base64 encoded string> }
```


In [ ]:
# GCS からサンプル画像をダウンロード。
!gcloud storage cp gs://asl-public-data/data/flowers/jpegs/10172567486_2748826a8b.jpg sample.jpg


def b64encode(filename):
    with open(filename, "rb") as ifp:
        img_bytes = ifp.read()
        return base64.b64encode(img_bytes).decode()


data = {
    "signature_name": "predict_base64",
    "instances": [{"img_bytes": {"b64": b64encode("./sample.jpg")}}],
}

Python SDK の `endpoint.predict` は `serving_default` シグネチャのみをサポートしているため、ここでは一般的な Python の `requests` モジュールで直接 API 呼び出しを定義し、代わりに他のシグネチャを指定できる `rawPredict` API を使用します。

これを行うには、認証トークンを定義し、リクエストヘッダーに設定する必要があります。


In [ ]:
token = (
    GoogleCredentials.get_application_default().get_access_token().access_token
)
headers = {"Authorization": "Bearer " + token}

エンドポイントの URL は `https://<region>-aiplatform.googleapis.com/v1/projects/<project id>/locations/<region>/endpoints/<endpoint id>:rawPredict` です。<br>
これに合わせて定義し、エンコードされた生の画像を API に送信しましょう。


In [ ]:
api = "https://{}-aiplatform.googleapis.com/v1/projects/{}/locations/{}/endpoints/{}:rawPredict".format(
    REGION, PROJECT, REGION, endpoint.name
)

response = requests.post(api, json=data, headers=headers)
json.loads(response.content)

これで、Agent Platform オンライン予測サービスから結果を取得できました。


## まとめ
以下の方法を学びました:
- SavedModel のエクスポートと読み込み
- サービング関数をカスタマイズして SavedModel の挙動を制御する
- SavedModel を Agent Platform にデプロイする
- デプロイされたモデルをバッチ推論とオンライン推論の両方で使用する


Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.